In [13]:
%load_ext rpy2.ipython

In [14]:
import pandas as pd
from sqlalchemy import Integer
from sqlalchemy import create_engine
from sqlalchemy import func
from sqlalchemy.orm import Query

import src
from src.data.models import Channel
from src.data.models import Comment
from src.data.models import Sentence
from src.data.models import Video

In [15]:
pd.options.display.float_format = "{:.1f}".format

colormap = pd.DataFrame(src.colormap.items(), columns=["channel", "color"])

engine = create_engine(src.PS_ENGINE)

# per Channel

In [16]:
query_all = (
    Query(Channel)
    .join(Video)
    .join(Sentence)
    .group_by(Channel)
    .filter(
        Video.format == "videos",
    )
    .with_entities(
        Channel.channel,
        Channel.channel_follower_count.label("followers"),
        func.count(Video.id.distinct()).label("videos"),
        func.count(Sentence.id).label("sentences"),
        func.min(Video.datetime_upload).label("first_video"),
        func.max(Video.datetime_upload).label("latest_video"),
    )
    .order_by(Channel.channel_follower_count.desc())
)



with engine.connect() as conn:
    df_all = pd.read_sql(query_all.statement, conn)

df_all.channel = df_all.channel.replace(
    {"BÜNDNIS 90/DIE GRÜNEN": "Grüne", "AfD-Fraktion Bundestag": "AfD BT"},
)
df_all

,channel,followers,videos,sentences,first_video,latest_video
0,AfD BT,388000,5257,318328,2017-12-06,2024-01-26
1,AfD TV,250000,1563,160132,2017-06-13,2024-01-26
2,DIE LINKE,29000,1499,157693,2008-12-18,2024-01-22
3,Grüne,26100,1623,119187,2008-05-07,2024-01-27
4,SPD,24200,1625,191038,2008-05-08,2024-01-24
5,FDP,23300,1952,159493,2007-09-06,2024-01-06
6,CDU,21900,2111,142666,2008-08-22,2024-01-27
7,CSU,5170,730,42547,2008-09-09,2023-10-05


In [17]:
query_after = (
    Query(Channel)
    .join(Video)
    .join(Sentence)
    .group_by(Channel)
    .filter(
        Video.is_valid == True,
        Sentence.is_valid == True,
    )
    .with_entities(
        Channel.channel,
        Channel.channel_follower_count.label("followers"),
        func.count(Video.id.distinct()).label("videos"),
        func.count(Sentence.id).label("sentences"),
        func.sum(Sentence.elite.cast(Integer)).label("sum_elite"),
        func.sum(Sentence.pplcentr.cast(Integer)).label("sum_pplcentr"),
        # func.min(Video.datetime_upload).label("first_video"),
        # func.max(Video.datetime_upload).label("latest_video"),
    )
    .order_by(Channel.channel_follower_count.desc())
)
with engine.connect() as conn:
    df_valid = pd.read_sql(query_after.statement, conn)

df_valid.channel = df_valid.channel.replace(
    {"BÜNDNIS 90/DIE GRÜNEN": "Grüne", "AfD-Fraktion Bundestag": "AfD BT"},
)

df_valid

,channel,followers,videos,sentences,sum_elite,sum_pplcentr
0,AfD BT,388000,5207,299414,40413,6117
1,AfD TV,250000,1457,145525,17477,3663
2,DIE LINKE,29000,435,46823,2499,1425
3,Grüne,26100,459,46964,1463,1358
4,SPD,24200,480,66999,1433,2253
5,FDP,23300,487,36930,1400,929
6,CDU,21900,629,54267,947,1450
7,CSU,5170,145,10463,290,239


# per Video

In [18]:
comments_query = (
    Query(Comment.id, func.count(Comment.id))
    .filter(Comment.video_id == Video.id, Comment.is_valid == True)
    .with_entities(func.count(Comment.id))
    .scalar_subquery()
)
query = (
    Query(Video)
    .join(Channel)
    .filter(Video.is_valid == True)
    .with_entities(
        Channel.channel,
        Video.datetime_upload,
        Video.like_count.label("likes"),
        Video.view_count.label("views"),
        Video.duration,
        comments_query.label("comments"),
    )
)

with engine.connect() as conn:
    df = pd.read_sql(query.statement, conn)
df.channel = df.channel.replace(
    {"BÜNDNIS 90/DIE GRÜNEN": "Grüne", "AfD-Fraktion Bundestag": "AfD BT", "DIE LINKE": "Linke"},
)

In [19]:
df_videos = df.groupby("channel").mean(numeric_only=True).stack().reset_index()

pivot_videos = pd.pivot(df_videos, index="channel", columns="level_1", values=0)
pivot_videos.columns = [f"mean_{col.lstrip('@')}" for col in pivot_videos.columns]
pivot_videos = pivot_videos.reset_index()

In [20]:
pivot_videos

,channel,mean_comments,mean_duration,mean_likes,mean_views
0,AfD BT,398.3,436.2,3867.4,45832.0
1,AfD TV,398.7,657.2,3627.1,43363.7
2,CDU,41.2,616.0,64.4,9624.1
3,CSU,7.5,442.2,35.6,22332.8
4,FDP,0.7,620.3,0.5,5874.0
5,Grüne,0.2,892.2,79.7,4537.7
6,Linke,45.0,832.4,255.8,10426.9
7,SPD,25.0,1088.2,102.1,5065.4


In [21]:
df_valid

,channel,followers,videos,sentences,sum_elite,sum_pplcentr
0,AfD BT,388000,5207,299414,40413,6117
1,AfD TV,250000,1457,145525,17477,3663
2,DIE LINKE,29000,435,46823,2499,1425
3,Grüne,26100,459,46964,1463,1358
4,SPD,24200,480,66999,1433,2253
5,FDP,23300,487,36930,1400,929
6,CDU,21900,629,54267,947,1450
7,CSU,5170,145,10463,290,239


In [22]:
summary_table = pd.merge(df_valid, pivot_videos, on="channel").T
summary_table.columns = [col.lstrip("@") for col in summary_table.iloc[0,:]]
summary_table = summary_table.iloc[1:,:]
summary_table = summary_table

In [23]:
summary_table

,AfD BT,AfD TV,Grüne,SPD,FDP,CDU,CSU
followers,388000,250000,26100,24200,23300,21900,5170
videos,5207,1457,459,480,487,629,145
sentences,299414,145525,46964,66999,36930,54267,10463
sum_elite,40413,17477,1463,1433,1400,947,290
sum_pplcentr,6117,3663,1358,2253,929,1450,239
mean_comments,398.3,398.7,0.2,25.0,0.7,41.2,7.5
mean_duration,436.2,657.2,892.2,1088.2,620.3,616.0,442.2
mean_likes,3867.4,3627.1,79.7,102.1,0.5,64.4,35.6
mean_views,45832.0,43363.7,4537.7,5065.4,5874.0,9624.1,22332.8


In [24]:
summary_table.to_latex(
    src.PATH / "overleaf/tables/summary.tex",
    float_format= "{:.1f}".format,
    escape=True,
    column_format="lrrrrrrr",
)

In [25]:
%%R -i df -i colormap -w 1000 -h 600

suppressMessages(library(tidyverse))
library(ggplot2)
library(ggeffects)
library(here)

options(scipen = 999)

cmap <- setNames(colormap$color, colormap$channel)

ggplot(df, aes(x=channel, y=views, fill=channel)) +
   geom_violin(draw_quantiles=c(0.25, 0.5, 0.75), alpha=0.7) +
   scale_y_continuous(trans="log10") +
   scale_color_manual(values=cmap, aesthetics=c("color", "fill")) +
   theme_ggeffects(
      base_family = "serif",
      base_size = 22
   ) +
   theme(
      axis.text.x=element_text(angle=20, hjust=1)
   ) +
   xlab("Channel") +
   ylab("log10(ViewCount)")


ggsave(here("overleaf/img/view_count_violin.pdf"))

Saving 13.9 x 8.33 in image


here() starts at /Users/lukas/git/ytpop
